## 1. Imports

In [1]:
import os
import subprocess

## 2. Check if data is available

In [2]:
print("Checking for required data:")
if(not(os.path.exists("../Data/dewiki-latest-pages-articles-multistream-index.txt"))):
    print("Index-File not found. Download:")
    subprocess.call(['sh', '../bin/download-and-unzip-index.sh'])
print("Index-File available at: ../Data/dewiki-latest-pages-articles-multistream-index.txt")

if(not(os.path.exists("../Data/dewiki-latest-pages-articles-multistream.xml"))):
    print("Articles-File not found. Download (this might take a while):")
    subprocess.call(['sh', '../bin/download-and-unzip-data.sh'])
print("Articles-File available at: ../Data/dewiki-latest-pages-articles-multistream.xml")

Checking for required data:
Index-File available at: ../Data/dewiki-latest-pages-articles-multistream-index.txt
Articles-File available at: ../Data/dewiki-latest-pages-articles-multistream.xml


In [3]:
import wikipediaapi

ModuleNotFoundError: No module named 'wikipediaapi'

In [ ]:
def print_categories(page):
    categories = page.categories
    for title in sorted(categories.keys()):
        print("%s: %s" % (title, categories[title]))

In [ ]:
wiki_wiki = wikipediaapi.Wikipedia('de')
print_categories(wiki_wiki.page('Johann Wolfgang von Goethe'))

In [2]:
import mwxml
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

In [6]:
dump = mwxml.Dump.from_file(open("../Data/dewiki-latest-pages-articles-multistream.xml"))

def dump_to_dataset(dump:mwxml.iteration.dump.Dump, n_samples:int=-1) -> tuple([np.ndarray, np.ndarray]):
    data = []
    label = []
    i = 0
    #pbar = tqdm(total= (n_samples if (n_samples>1) else 50000000))
    for page in dump:
        for revisions in page:
            try:
                if("Liste von Autoren" not in revisions.page.title):
                    i += 1
                    if(re.search(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", revisions.text)):
                        label.append(1)
                        data.append(re.sub(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", '',revisions.text))
                    else:
                        label.append(0)
                        data.append(revisions.text)
                    #pbar.update(1)
            except Exception as e:
                print(e)
        if(i>=n_samples):
            break
    #pbar.close()
    return np.array(data), np.array(label)

In [7]:
features, labels = dump_to_dataset(dump, 10000)

In [8]:
print(len(features), len(labels))

10000 10000


In [31]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [32]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  1


In [33]:
tokenizer = Tokenizer(
    num_words=10000,
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
)
tokenizer.fit_on_texts(features)

KeyboardInterrupt: 

In [12]:
sequences = tokenizer.texts_to_sequences(features)

In [13]:
padded_sequences = pad_sequences(sequences, maxlen=10000)

In [30]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=10000, output_dim=64),
    tf.keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(3, activation='softmax')
])
opt = tf.keras.optimizers.legacy.Adam(learning_rate=0.0001)

model.compile(optimizer=opt, loss=["sparse_categorical_crossentropy"], weighted_metrics=['accuracy'])

In [35]:
earlystopper = EarlyStopping(patience=5, restore_best_weights=True, verbose=1)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00000001, verbose=1, cooldown=30)

In [18]:
history = model.fit(
    features,
    labels,
    epochs=100,
    batch_size=50,
    verbose=2, 
    shuffle=True,
    callbacks=[earlystopper, reduce_lr]
)

AttributeError: 'History' object has no attribute 'metrics'